In [1]:
import os
import time
import json

from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

import sys
import json
import subprocess
from pathlib import Path

In [2]:
load_dotenv(find_dotenv())

# OpenAI API configuration
openai_key = os.getenv("OPENAI_API_KEY")

assert openai_key, "OPENAI_API_KEY not found in .env"

# Cloud model
openai_model = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=openai_key,
    temperature=0
)

# Local Ollama model
ollama_base_url = os.getenv(
    "OLLAMA_BASE_URL",
    "http://127.0.0.1:11434"
)

ollama_model_name = os.getenv(
    "OLLAMA_MODEL",
    "qwen2.5:3b"
)

ollama_model = ChatOpenAI(
    model=ollama_model_name,
    base_url=f"{ollama_base_url}/v1",
    api_key="ollama",
    temperature=0
)

print(f"Ollama model configured: {ollama_model_name}")
print(f"Ollama endpoint: {ollama_base_url}")

Ollama model configured: qwen2.5:3b
Ollama endpoint: http://127.0.0.1:11434


In [3]:
SERVER_PATH = Path("recon_server.py").resolve()
assert SERVER_PATH.exists(), f"Could not find {SERVER_PATH}"

def extract_text(result) -> str:
    if getattr(result, "content", None):
        parts = [item.text for item in result.content if hasattr(item, "text")]
        if parts:
            return "\n".join(parts).strip()
    return str(result).strip()

async def discover_and_run_initial_scans():
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(SERVER_PATH)]
    )

    async with stdio_client(server_params, errlog=subprocess.DEVNULL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools_result = await session.list_tools()
            available_tools = [
                {
                    "name": tool.name,
                    "description": tool.description,
                    "input_schema": tool.inputSchema
                }
                for tool in tools_result.tools
            ]
            discovered_names = {t["name"] for t in available_tools}

            # 1. Demonstrate fail-closed out-of-scope refusal before running lab scans
            refusal_raw = await session.call_tool("banner_grab", {"target": "8.8.8.8", "port": 80})
            scope_refusal = extract_text(refusal_raw)
            print("[OUT-OF-SCOPE GUARDRAIL TEST]:", scope_refusal)

            # 2. Gate initial in-scope scans behind typed APPROVE
            initial_scans = {
                "scan_http_headers": {"target": "127.0.0.1", "port": 3000},
                "enumerate_http_methods": {"target": "127.0.0.1", "port": 3000},
                "analyze_cookie_security": {"target": "127.0.0.1", "port": 3000},
                "banner_grab": {"target": "127.0.0.1", "port": 2121},
                "anonymous_ftp_check": {"target": "127.0.0.1", "port": 2121},
                "http_probe": {"target": "127.0.0.1", "port": 8080, "path": "/"},
            }

            print("\nProposed in-scope scans on 127.0.0.1:")
            for name, args in initial_scans.items():
                print(f"  - {name}: {args}")

            approval = input("\nType APPROVE to run initial in-scope scanners: ").strip()
            if approval != "APPROVE":
                raise RuntimeError("Initial reconnaissance cancelled by human gate.")

            scanner_results = {}
            for name, args in initial_scans.items():
                if name not in discovered_names:
                    scanner_results[name] = f"SKIPPED: '{name}' not found on MCP server"
                    continue
                res = await session.call_tool(name, args)
                scanner_results[name] = extract_text(res)
                print(f"Completed: {name}")

            return available_tools, scope_refusal, scanner_results

available_tools, scope_refusal, scanner_results = await discover_and_run_initial_scans()
tool_descriptions = json.dumps(available_tools, indent=2)
combined_findings = json.dumps(scanner_results, indent=2)

print("\nCollected findings keys:", list(scanner_results.keys()))

[OUT-OF-SCOPE GUARDRAIL TEST]: REFUSED: Target '8.8.8.8' is outside the authorized allowlist ['127.0.0.1'].

Proposed in-scope scans on 127.0.0.1:
  - scan_http_headers: {'target': '127.0.0.1', 'port': 3000}
  - enumerate_http_methods: {'target': '127.0.0.1', 'port': 3000}
  - analyze_cookie_security: {'target': '127.0.0.1', 'port': 3000}
  - banner_grab: {'target': '127.0.0.1', 'port': 2121}
  - anonymous_ftp_check: {'target': '127.0.0.1', 'port': 2121}
  - http_probe: {'target': '127.0.0.1', 'port': 8080, 'path': '/'}
Completed: scan_http_headers
Completed: enumerate_http_methods
Completed: analyze_cookie_security
Completed: banner_grab
Completed: anonymous_ftp_check
Completed: http_probe

Collected findings keys: ['scan_http_headers', 'enumerate_http_methods', 'analyze_cookie_security', 'banner_grab', 'anonymous_ftp_check', 'http_probe']


In [4]:
selection_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an AI security analyst performing authorized,
read-only reconnaissance.

You have access to the following MCP tools:
{tools}

Rules:
1. Only recommend tools that appear in the provided tool list.
2. Only use the authorized target 127.0.0.1.
3. Web tools must use port 3000.
4. Service tools must use ports 2121 or 8080.
5. Do not recommend exploitation or destructive actions.
6. Select a tool only when additional investigation is justified.
7. If no additional investigation is justified, return NONE.
8. Do not execute tools.

Return JSON only, using this structure:

{{
    "tool_name": "tool name or NONE",
    "arguments": {{}},
    "reason": "Why this tool was selected"
}}
"""),
    ("user", """
Analyze these security findings and recommend the
most appropriate next investigation:

{findings}
""")
])

openai_selection_chain = selection_prompt | openai_model | StrOutputParser()
ollama_selection_chain = selection_prompt | ollama_model | StrOutputParser()

start_time = time.perf_counter()
openai_selection = openai_selection_chain.invoke({
    "tools": tool_descriptions,
    "findings": combined_findings
})
openai_time = time.perf_counter() - start_time

start_time = time.perf_counter()
ollama_selection = ollama_selection_chain.invoke({
    "tools": tool_descriptions,
    "findings": combined_findings
})
ollama_time = time.perf_counter() - start_time

print("OPENAI RECOMMENDATION")
print(openai_selection)
print(f"Time: {openai_time:.2f} seconds")

print("\nOLLAMA RECOMMENDATION")
print(ollama_selection)
print(f"Time: {ollama_time:.2f} seconds")

OPENAI RECOMMENDATION
```json
{
    "tool_name": "analyze_cookie_security",
    "arguments": {
        "target": "127.0.0.1",
        "port": 3000
    },
    "reason": "The analysis shows that no cookies were found in the response. Investigating cookie security is justified to ensure that if cookies are implemented in the future, they will have appropriate security attributes."
}
```
Time: 1.80 seconds

OLLAMA RECOMMENDATION
{
    "tool_name": "analyze_cookie_security",
    "arguments": {},
    "reason": "The analyze_cookie_security tool was selected because it found no cookies in the response, which could indicate potential security issues related to cookie management."
}
Time: 6.16 seconds


In [ ]:
import re

cve_kb_text = Path("cve_knowledge_base.md").read_text(encoding="utf-8") if Path("cve_knowledge_base.md").exists() else ""
web_kb_text = Path("web_security_kb.md").read_text(encoding="utf-8") if Path("web_security_kb.md").exists() else ""
combined_kb_text = f"=== SERVICE KB ===\n{cve_kb_text}\n\n=== WEB KB ===\n{web_kb_text}"

def parse_model_json(raw_text: str) -> dict:
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_text.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"tool_name": "NONE", "arguments": {}, "reason": f"Unparsed model output: {raw_text}"}

async def discover_tool_names() -> set[str]:
    server_params = StdioServerParameters(command=sys.executable, args=[str(SERVER_PATH)])
    async with stdio_client(server_params, errlog=subprocess.DEVNULL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_res = await session.list_tools()
            return {t.name for t in tools_res.tools}

async def run_gated_agent_followup(recommendation_text: str):
    discovered = await discover_tool_names()
    rec = parse_model_json(recommendation_text)
    proposed_tool = rec.get("tool_name", "NONE")
    proposed_args = dict(rec.get("arguments") or {})

    if proposed_tool == "NONE" or proposed_tool not in discovered:
        print(f"[AGENT DECISION] No additional tool executed (tool_name={proposed_tool!r}). Reason: {rec.get('reason')}")
        return f"No follow-up tool executed (selection={proposed_tool})."

    # Enforce authorized 127.0.0.1 target & sensible defaults if model left arguments empty
    if "target" in proposed_args and proposed_args["target"] != "127.0.0.1":
        raise ValueError(f"Blocked unapproved target in recommendation: {proposed_args['target']}")
    proposed_args.setdefault("target", "127.0.0.1")
    if proposed_tool == "nmap_service_scan":
        proposed_args.setdefault("ports", "2121,8080")
    elif proposed_tool in {"scan_http_headers", "enumerate_http_methods", "analyze_cookie_security"}:
        proposed_args.setdefault("port", 3000)

    print(f"[AGENT PROPOSED ACTION] {proposed_tool} with args={proposed_args}")
    print(f"Rationale: {rec.get('reason')}")

    # 1. Demonstrate non-APPROVE refusal/block behavior
    sample_reject = input("Test non-APPROVE gate first (type anything other than APPROVE, e.g. SKIP): ").strip()
    if sample_reject != "APPROVE":
        print(f"[BLOCKED - NO EXECUTION] Input '{sample_reject}' != 'APPROVE'. Tool '{proposed_tool}' did NOT run.")

    # 2. Prompt for actual approval to execute
    decision = input("Type APPROVE to execute this agent-selected follow-up scan: ").strip()
    if decision != "APPROVE":
        print("[NOT APPROVED] Follow-up tool was not executed.")
        return "Follow-up scan not executed (approval not typed)."

    server_params = StdioServerParameters(command=sys.executable, args=[str(SERVER_PATH)])
    async with stdio_client(server_params, errlog=subprocess.DEVNULL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            res = await session.call_tool(proposed_tool, proposed_args)
            return extract_text(res)

followup_result = await run_gated_agent_followup(openai_selection)
print("\nFollow-up scan output:\n", followup_result)

report_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an AI security analyst producing a single combined report across service and web findings.
Ground every claim strictly in the observed scan evidence and the provided knowledge bases.
Do not treat missing cookies or no advertised OPTIONS methods as vulnerabilities unless supported by evidence.
Include CVE, CWE, or OWASP mappings where supported, give severity and remediation, and label unverified/missing checks as GAP.

Knowledge Bases:
{kb}
"""),
    ("user", """
Initial Scan Results:
{findings}

Follow-Up Scan Output:
{followup}
""")
])

combined_report = (report_prompt | openai_model | StrOutputParser()).invoke({
    "kb": combined_kb_text,
    "findings": combined_findings,
    "followup": followup_result
})

Path("results").mkdir(parents=True, exist_ok=True)
Path("results/combined_grounded_report.md").write_text(combined_report, encoding="utf-8")
Path("results/model_comparison.json").write_text(json.dumps({
    "openai_model": openai_model_name if "openai_model_name" in globals() else "gpt-4o-mini",
    "ollama_model": ollama_model_name if "ollama_model_name" in globals() else "qwen2.5:3b",
    "openai_time_sec": round(openai_time, 2),
    "ollama_time_sec": round(ollama_time, 2),
    "openai_selection": parse_model_json(openai_selection),
    "ollama_selection": parse_model_json(ollama_selection),
}, indent=2), encoding="utf-8")

print("\n=== COMBINED GROUNDED REPORT ===\n")
print(combined_report)

[HUMAN GATE] Proposed action: nmap_service_scan with args={'target': '127.0.0.1', 'ports': '2121,8080'}

Follow-up scan output:
 Starting Nmap 7.80 ( https://nmap.org ) at 2026-11-27 15:44 Eastern Standard Time
Nmap scan report for kubernetes.docker.internal (127.0.0.1)
Host is up (0.00s latency).

PORT     STATE SERVICE VERSION
2121/tcp open  ftp     vsftpd 2.3.4
8080/tcp open  http    Apache httpd 2.2.8 ((Ubuntu) DAV/2)
Service Info: OS: Unix

Service detection performed. Please report any incorrect results at https://nmap.org/submit/ .
Nmap done: 1 IP address (1 host up) scanned in 17.02 seconds

=== COMBINED GROUNDED REPORT ===

# Combined Security Report

## Summary
This report consolidates findings from both service and web scans conducted on the target system. The analysis reveals several critical vulnerabilities and misconfigurations that require immediate attention.

---

## Service Findings

### 1. vsFTPd 2.3.4 Backdoor
- **Service / Banner:** `vsFTPd 2.3.4` (Port 2121/tcp)
-